# M01 · Center-aligned PWM, dead time and input capture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/04_control_motion/M01_center_aligned_pwm_dead_time_and_input_capture.ipynb)

**Track:** Control and motion · **Skill axis (sheet):** Control and motion · **Time:** about 55 min

**Skill Hub concepts:** Center-aligned PWM and dead time; Input capture

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("M01")

## Map: the timer as a waveform machine

```
 edge-aligned:   counter /|/|/|      output high while CNT < CCR       f = f_tim / (ARR + 1)
 center-aligned: counter /\/\/\      output high while CNT < CCR       f = f_tim / (2 ARR)

 half-bridge with dead time:
 reference   ____|‾‾‾‾‾‾‾‾‾‾‾‾|______
 high side   ______|‾‾‾‾‾‾‾‾‾‾|______     rising edge delayed by t_dead
 low side    ‾‾‾|________________|‾‾‾     rising edge delayed by t_dead
                ^^ both off ^^     ^^ both off
```

| Choice | Why |
|---|---|
| Center-aligned for motor drives | switching edges are symmetric; the period center is quiet, so the ADC samples phase current there |
| Dead time | MOSFETs turn off slower than they turn on; without a gap both conduct (shoot-through) and short the supply |
| Dead-time cost | output voltage error of about $\pm V_{dc}\, t_{dead} / T_{pwm}$ depending on current direction |

**Input capture:** on an edge the timer copies its counter into a capture register. Period in ticks between two
captures on a counter that wraps at $\mathrm{ARR}+1$: $\Delta = (c_2 - c_1) \bmod (\mathrm{ARR}+1)$;
$f = f_{tim}/\Delta$, duty $= \big((c_{fall} - c_{rise}) \bmod (\mathrm{ARR}+1)\big)/\Delta$.

## Task 1: generate complementary outputs with dead time
Simulate tick by tick. The counter goes up 0..ARR then down ARR..1 (period = 2 ARR ticks). The reference is 1 while
`counter < ccr`. The high side copies the reference but each 0-to-1 edge is delayed by `dead` ticks; the low side
copies the inverted reference with the same delay on its 0-to-1 edges. Return `(high, low)` lists.

In [ ]:
def complementary_pwm(arr, ccr, dead, periods=2):
    raise NotImplementedError  # TODO

lab.check("pwm", complementary_pwm)

In [ ]:
%%writefile m01_pwm.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/m01.h
// Grade it with lab.check("pwm", "m01_pwm.c"). For C++, name the file .cpp in both places.
#include "m01.h"

size_t complementary_pwm(uint32_t arr, uint32_t ccr, uint32_t dead, int periods, uint8_t *hi, uint8_t *lo, size_t cap) {
    (void)ccr; (void)dead;
    size_t n = 2u * arr * (size_t)periods;
    if (n > cap) n = cap;
    memset(hi, 0, n); memset(lo, 0, n);
    return n; /* TODO: up-down count, compare with ccr, delay each rising edge by `dead` counts */
}

In [ ]:
# lab.check("pwm", "m01_pwm.c")  # uncomment after writing the C/C++ cell above

## Task 2: measure frequency and duty from captures

In [ ]:
def capture_measure(rise1, fall, rise2, arr, f_tim):
    """Return (frequency_hz, duty) from rising, falling, next rising capture values."""
    raise NotImplementedError  # TODO

lab.check("capture", capture_measure)

In [ ]:
%%writefile m01_capture.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/m01.h
// Grade it with lab.check("capture", "m01_capture.c"). For C++, name the file .cpp in both places.
#include "m01.h"

void capture_measure(uint32_t rise1, uint32_t fall, uint32_t rise2, uint32_t arr, double f_tim, double *freq_hz, double *duty) {
    (void)rise1; (void)fall; (void)rise2; (void)arr; (void)f_tim;
    *freq_hz = 0; *duty = 0; /* TODO: differences modulo arr + 1 */
}

In [ ]:
# lab.check("capture", "m01_capture.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** 170 MHz timer, center-aligned, ARR = 4250. PWM frequency in Hz?

**M2 (Apply).** Dead time of 500 ns at 170 MHz, in timer ticks?

**M3 (Analyze).** 1 MHz capture timer, ARR = 65535. Rising edges captured at 65000, then 1464. Signal frequency in Hz?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Why sample motor phase current at the center of a center-aligned PWM period?
- a) the counter is at zero
- b) no switch is changing state there, so the current waveform is free of switching ringing and at its average value
- c) it doubles the ADC resolution

**K2 (Analyze).** Explain shoot-through, how dead time prevents it, and the side effect dead time has on the voltage
the motor actually sees (and how firmware compensates).

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `m01.h`).
3. In `roboprep/checks/m01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()